## Overview of Assignment 4

This assignment focuses on exploring and implementing advanced concepts and techniques in information retrieval. The primary objectives are to build Retrieval Augumentation Generation, and learn about Language Models

## Enter your details below

## Name

MD SANIM FARHAN

## Banner ID

B00906667

## GitHub Link of your Assingment 4

## Q1 : Setting up the libraries and the environment

In [2]:
# Q1(a) - Install required libraries

!pip install transformers
!pip install datasets
!pip install peft
!pip install accelerate
!pip install bitsandbytes
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install sentence-transformers
!pip install faiss-cpu

## Q2:  Data Preprocessing and Model Selection

In [3]:
import os
import tempfile

# Create a temporary directory for the dataset
docs_dir = tempfile.mkdtemp()

# Sample dataset about Artificial Intelligence and Machine Learning
documents = {
    "machine_learning.txt": """
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.
""",

    "deep_learning.txt": """
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.
""",

    "rag.txt": """
Retrieval-Augmented Generation, also called RAG, combines information
retrieval with large language models.

In a RAG system, documents are divided into smaller chunks and converted
into vector embeddings. When a user asks a question, the system retrieves
relevant chunks and provides them to the language model as context.

RAG can help language models answer questions using information stored
in external documents.
"""
}

# Save documents as text files
for filename, content in documents.items():
    file_path = os.path.join(docs_dir, filename)

    with open(file_path, "w", encoding="utf-8") as file:
        file.write(content)

print(f"Created {len(documents)} documents.")
print(f"Dataset directory: {docs_dir}")

Created 3 documents.
Dataset directory: C:\Users\sanim\AppData\Local\Temp\tmpy5y9zbaz


In [4]:
# we use the Textloader and load the documents

In [5]:
from langchain_community.document_loaders import TextLoader 
all_documents = []

for filename in documents.keys():
    file_path = os.path.join(docs_dir,filename)

    loader = TextLoader(file_path)
    loaded_docs = loader.load()

    all_documents.extend(loaded_docs)

print(f"Loaded {len(all_documents)} documets")

C:\Users\sanim\AppData\Local\Temp\ipykernel_14872\3588072547.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loaded 3 documets


In [6]:
# Split the text into chunks

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter =RecursiveCharacterTextSplitter(
    chunk_size =500,
    chunk_overlap=50,
    separators = ["\n\n", "\n", ".", " ", ""]
)

document_chunks = text_splitter.split_documents(all_documents)
print(f"Created {len(document_chunks)} document chunks.")

print(f"\n Sample chunk:")
print(document_chunks[0].page_content)

print("\nMetadata: ")
print(document_chunks[0].metadata)

Created 3 document chunks.

 Sample chunk:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

Metadata: 
{'source': 'C:\\Users\\sanim\\AppData\\Local\\Temp\\tmpy5y9zbaz\\machine_learning.txt'}


### Q2.1 Dataset Preprocessing

A small text dataset containing information about artificial intelligence,
machine learning, deep learning, and RAG is used. The documents are loaded
using TextLoader and divided into smaller chunks using
RecursiveCharacterTextSplitter. A chunk size of 500 and an overlap of 50
are used to preserve context between neighbouring chunks.

In [8]:
#loading tokenizer 

from transformers import AutoTokenizer 

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

C:\Users\sanim\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanim\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [9]:
# Tokenize the document chunks 
tokenized_chunks = []

for chunk in document_chunks:
    tokens = tokenizer(
        chunk.page_content,
        truncation=True,
        padding=False,
        max_length=512,
        return_tensors="pt"
    )

    tokenized_chunks.append(tokens)

print(f"Tokenized {len(tokenized_chunks)} chunks.")


Tokenized 3 chunks.


In [10]:
# Showing that tokenization worked 
print("Original text:")
print(document_chunks[0].page_content)

print("\nToken IDs:")
print(tokenized_chunks[0]["input_ids"])

print("\nNumber of tokens:")
print(tokenized_chunks[0]["input_ids"].shape[1])

Original text:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

Token IDs:
tensor([[    1,  6189,  6509,   338,   263,  5443,   310, 23116, 21082,   393,
          6511, 23226,    13,   517,  5110, 15038,   515,   848,  1728,  1641,
          9479,  1824,  2168, 29889,    13,    13,  8439,   526,  2211,  3619,
          4072,   310,  4933,  6509, 29901,  2428, 11292,  6509, 29892,    13,
           348,  9136, 11292,  6509, 29892,   322, 15561,  1454, 13561,  6509,
         29889,    13,    13, 19111, 11292,  6509,  3913,  3858,   839,   848,
           304,  7945,  4733, 29889, 13103,  2428, 11292,    13, 21891,  9595,
          3160, 12965,   

### Q2.2 Tokenization

The TinyLlama tokenizer is used because TinyLlama is the pretrained
language model selected for the RAG system. The tokenizer converts each
text chunk into token IDs that can be understood by the language model.
The maximum sequence length is set to 512 tokens and truncation is enabled
to prevent sequences from exceeding this limit.

In [11]:
# Q2.3 - Split tokenized data into chunks for indexing

token_chunks = []

chunk_size = 256
chunk_overlap = 50

for tokens in tokenized_chunks:
    input_ids = tokens["input_ids"][0]

    start = 0

    while start < len(input_ids):
        end = start + chunk_size

        chunk = input_ids[start:end]

        token_chunks.append(chunk)

        if end >= len(input_ids):
            break

        start += chunk_size - chunk_overlap

print(f"Created {len(token_chunks)} token chunks.")

Created 3 token chunks.


### Q2.3 Token Chunking

The tokenized documents are divided into smaller chunks of 256 tokens with
an overlap of 50 tokens. The overlap helps preserve context between consecutive
chunks. These smaller chunks can later be converted into embeddings and stored
in a vector index for efficient retrieval.

In [12]:
# Convert token chunks back into text for embedding
indexing_chunks = []

for chunk in token_chunks:
    text = tokenizer.decode(chunk, skip_special_tokens=True)
    indexing_chunks.append(text)

print(f"Prepared {len(indexing_chunks)} chunks for indexing.")
print("\nSample chunk:")
print(indexing_chunks[0])

Prepared 3 chunks for indexing.

Sample chunk:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.


In [14]:
# Create embeddings
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model.encode(indexing_chunks)

print("Embedding shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (3, 384)


In [15]:
import faiss
import numpy as np

# Convert embeddings to float32, which FAISS expects
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

# Get embedding dimension
dimension = chunk_embeddings.shape[1]

# Create FAISS L2 index
index = faiss.IndexFlatL2(dimension)

# Add embeddings to the index
index.add(chunk_embeddings)

print(f"FAISS index created successfully.")
print(f"Number of vectors stored: {index.ntotal}")

FAISS index created successfully.
Number of vectors stored: 3


In [16]:
query = "What is machine learning?"

query_embedding = embedding_model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

distances, indices = index.search(query_embedding, k=2)

print("Query:", query)

for i, idx in enumerate(indices[0]):
    print(f"\nResult {i + 1}:")
    print(indexing_chunks[idx])
    print("Distance:", distances[0][i])

Query: What is machine learning?

Result 1:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.
Distance: 0.43365866

Result 2:
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.
Distance: 0.9418586


### Q2.4 Vector Store

The `all-MiniLM-L6-v2` sentence-transformer model is used to convert the text chunks into numerical embedding vectors. These embeddings are stored in a FAISS `IndexFlatL2` vector index. FAISS allows relevant chunks to be efficiently retrieved by comparing the embedding of a user's query with the stored document embeddings using L2 distance.


## Q3: Implementing RAG using LangChain for different queries

## Q4 : Modify and evaluate the different components of RAG

## Q5: Selecting and implementing a pretrained model for a new task